# 05 — Model, bobot kelas, dan fitting akhir

**File:** `05_Model_dan_Bobot.ipynb`

Kita menelusuri bagaimana satu kolom target menjadi satu classifier XGBoost/LightGBM. Ada **109 classifier biner per kandidat**; setiap classifier bisa berisi banyak pohon. Di sini kita membuat objek model tanpa melatihnya, memeriksa bobot train, lalu membaca daftar model final.

**Input:** target train beku, konfigurasi, dan recipe delapan kandidat tersimpan. **Hubungan tahap:** 04: split train/test dan fold → notebook ini → 06: Optuna, CV, dan OOF.

**Cara menjalankan:** pilih kernel `.venv` project, lalu **Restart Kernel and Run All**. Setiap notebook memuat datanya sendiri; tidak memerlukan variabel notebook lain. Source yang tampil dibaca langsung dari fungsi dalam `KODE_ML/research/` yang di-import, bukan salinan contoh.

**Batas eksekusi:** Run All bawaan membaca hasil utama yang tersimpan dan menjalankan pemeriksaan ringan. Menampilkan definisi fungsi tidak menjalankan training di dalamnya. Hasil lama dan hasil eksekusi pemeriksaan dibedakan pada tiap blok. Training baru hanya melalui gerbang eksplisit di notebook 07, yang default-nya mati.

## Persiapan: pastikan sumber berasal dari KODE_ML

Blok ini mencari folder sumber, menambahkannya ke jalur import, dan memeriksa alamat modul. `sys.dont_write_bytecode` mencegah penulisan cache Python saat belajar. Output alamat membantu mendeteksi bila kita tidak sengaja membuka salinan project lain.

In [1]:
from pathlib import Path
import sys

lokasi = Path.cwd().resolve()
kandidat = [p for induk in [lokasi, *lokasi.parents]
            for p in (induk, induk / "KODE_ML")]
kandidat.append(Path(r"C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML"))
KODE_ML = next((p for p in kandidat if (p / "research/runtime.py").is_file()), None)
if KODE_ML is None:
    raise FileNotFoundError("Buka notebook dari folder KODE_ML yang lengkap.")
sys.dont_write_bytecode = True
sys.path.insert(0, str(KODE_ML))
from IPython.display import display, Markdown
from belajar import tampilkan_sumber, tampilkan_awal
from research import runtime
assert Path(runtime.__file__).resolve().is_relative_to(KODE_ML.resolve())
print("Source aktif:", Path(runtime.__file__).resolve().parent)

Source aktif: C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML\research


### Baca konfigurasi dan data train

`verify()` memeriksa file/skema/identitas baris. `load_partition("train")` mengambil fitur aktual dan target pada indeks train yang sama. Nama label dibaca dari manifest; angka 109 tidak dibuat dalam sel. Label test belum dipakai untuk menentukan parameter.

In [2]:
import numpy as np
import pandas as pd
from research import training, dataset, experiment
from research.results import main_run

verified = dataset.verify()
meta = verified[0]
train = dataset.load_partition("train", verified=verified)
labels = meta["labels"]
protocol = runtime.read(experiment.PROTOCOL)
run = main_run()
print({"baris_train": len(train["truth"]), "label": len(labels),
       "morgan": train["morgan"].shape, "deskriptor": train["descriptors"].shape})

{'baris_train': 5353, 'label': 109, 'morgan': (5353, 2048), 'deskriptor': (5353, 8)}


**Cara baca:** baris = molekul, kolom target = label aroma. A/C memakai Morgan 2048; B/D memakai 2056 fitur. Satu target molekul boleh mengandung beberapa angka 1.

### Import dan konstanta `training.py`

Library XGBoost/LightGBM membangun estimator. `runtime` menyediakan profil CPU, batas proses, baca/tulis checkpoint, serta checksum. Semua fungsi training di bawah memakai import yang sama.

In [3]:
tampilkan_awal(training)

**Import dan konstanta aktual:** `KODE_ML/research/training.py`. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
"""Satu classifier biner per label; bobot hanya dari bagian latih."""
import joblib
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import xgboost
from .runtime import compute_profile, Limits, read, write, digest, WEIGHTING_POLICY

```

### `training.make_model` — kode aktual

**File:** `KODE_ML/research/training.py`.

**Input:** algoritma dan dictionary parameter. **Output:** estimator yang belum dilatih.

- Salinan `params` mencegah konfigurasi pemanggil ikut berubah.
- `compute_profile()` menentukan jumlah thread dan metode pohon XGBoost.
- Kedua algoritma memakai seed model 42. Ini berbeda peran dari seed pembagian fold.
- `**params` memasukkan recipe baseline atau hasil Optuna. Parameter lain menggunakan default library yang versinya dikunci.

Fungsi ini belum memanggil `.fit()` sehingga aman dipakai untuk melihat konfigurasi.

Sel berikut menampilkan definisi lengkap beserta lokasi dan hash sumber. Isinya dipakai oleh pipeline, bukan implementasi kedua.

In [4]:
tampilkan_sumber(training.make_model)

**Source aktual:** `KODE_ML/research/training.py`, baris 9–18. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
def make_model(algorithm, parameters):
    params = dict(parameters)
    profile = compute_profile()
    threads = profile["threads_per_model"]
    if algorithm == "xgb":
        return XGBClassifier(n_jobs=threads, random_state=42, eval_metric="logloss",
                             tree_method=profile["xgboost_tree_method"], **params)
    if algorithm == "lgbm":
        return LGBMClassifier(n_jobs=threads, random_state=42, verbosity=-1, **params)
    raise ValueError("Unknown algorithm.")

```

### Periksa delapan recipe aktual

Kita membuat objek estimator dari recipe hasil utama, tanpa fitting. Tabel menunjukkan fitur, seed model, dan parameter yang benar-benar disimpan.

In [5]:
rows = []
for key in experiment.CANDIDATES:
    recipe = runtime.read(run / key / "recipe.json")
    estimator = training.make_model(key.split("_")[0], recipe)
    rows.append({"kandidat": key, "fitur": 2048 + (8 if key[-1] in "BD" else 0),
                 "tuned": key in experiment.TUNED,
                 "seed_model": estimator.get_params()["random_state"], **recipe})
display(pd.DataFrame(rows).set_index("kandidat"))

,fitur,tuned,seed_model,n_estimators,learning_rate,max_depth,colsample_bytree,subsample,reg_alpha,reg_lambda,min_child_weight,gamma,min_child_samples,num_leaves,subsample_freq
kandidat,,,,,,,,,,,,,,,
xgb_A,2048,False,42,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
xgb_B,2056,False,42,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
xgb_C,2048,True,42,1200,0.005394,8.0,0.882710,0.684936,0.000019,0.008261,2.814509,2.623782,NaN,NaN,NaN
xgb_D,2056,True,42,1200,0.005394,8.0,0.882710,0.684936,0.000019,0.008261,2.814509,2.623782,NaN,NaN,NaN
lgbm_A,2048,False,42,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lgbm_B,2056,False,42,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lgbm_C,2048,True,42,400,0.028581,6.0,0.740236,0.875156,0.092028,5.274218,NaN,NaN,16.0,29.0,1.0
lgbm_D,2056,True,42,1100,0.007598,7.0,0.393844,0.896796,0.000038,0.267942,NaN,NaN,41.0,75.0,1.0


### `training.balanced_sample_weights` — kode aktual

**File:** `KODE_ML/research/training.py`.

**Input:** satu target label dari bagian latih. **Output:** bobot setiap baris.

Pemeriksaan awal menolak target kosong/nonbiner/satu kelas. `counts` menghitung 0 dan 1. Rumus `n / (2 * counts)` memberi bobot lebih besar pada kelas yang sedikit; `per_class[values]` menempelkan bobot tersebut ke setiap baris.

Saat CV, inputnya **train fold**, bukan seluruh dataset. Saat final fit, inputnya seluruh outer train. Dengan begitu, proporsi validation/test tidak ikut menentukan bobot.

Sel berikut menampilkan definisi lengkap beserta lokasi dan hash sumber. Isinya dipakai oleh pipeline, bukan implementasi kedua.

In [6]:
tampilkan_sumber(training.balanced_sample_weights)

**Source aktual:** `KODE_ML/research/training.py`, baris 21–30. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
def balanced_sample_weights(y):
    """Use the same fold-local weighting rule for XGBoost and LightGBM."""
    values = np.asarray(y, dtype=np.int8)
    if values.ndim != 1 or len(values) == 0 or not np.isin(values, [0, 1]).all():
        raise ValueError("Expected a non-empty binary target vector.")
    counts = np.bincount(values, minlength=2)
    if np.any(counts == 0):
        raise ValueError("Balanced weighting requires both classes.")
    per_class = len(values) / (2.0 * counts)
    return per_class[values]

```

### Buktikan rumus bobot pada label aktual

Label pertama dalam manifest dipakai sebagai contoh. Total bobot dua kelas harus sama meskipun jumlah barisnya berbeda. Ini bukan resampling atau perubahan target.

In [7]:
y = train["truth"][:, 0]
weights = training.balanced_sample_weights(y)
table = pd.DataFrame([
    {"label": labels[0], "kelas": v, "baris": int((y == v).sum()),
     "bobot_per_baris": float(weights[y == v][0]), "total_bobot": float(weights[y == v].sum())}
    for v in (0, 1)
])
assert np.isclose(table.iloc[0].total_bobot, table.iloc[1].total_bobot)
display(table)

,label,kelas,baris,bobot_per_baris,total_bobot
0,acidic,0,5317,0.503385,2676.5
1,acidic,1,36,74.347222,2676.5


### `training.fit` — kode aktual

**File:** `KODE_ML/research/training.py`.

**Input:** algoritma, recipe, fitur `x`, satu kolom target `y`, batas eksekusi. **Output:** satu classifier terlatih.

Urutannya: cek batas → pastikan dua kelas → buat model → hitung bobot train → fit → cek batas lagi. Callback tiap iterasi memeriksa batas waktu/proses, **bukan early stopping berdasarkan validation**. Callback XGBoost dibersihkan setelah fitting agar tidak ikut tersimpan.

Notebook ini hanya menampilkan fungsi. Pelatihan penuh memakai runner pada notebook 07.

Sel berikut menampilkan definisi lengkap beserta lokasi dan hash sumber. Isinya dipakai oleh pipeline, bukan implementasi kedua.

In [8]:
tampilkan_sumber(training.fit)

**Source aktual:** `KODE_ML/research/training.py`, baris 33–53. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
def fit(algorithm, parameters, x, y, limits):
    limits.check(force=True)
    if np.unique(y).size != 2:
        raise ValueError("Training requires both classes.")
    model = make_model(algorithm, parameters)
    weights = balanced_sample_weights(y)
    if algorithm == "xgb":
        class Callback(xgboost.callback.TrainingCallback):
            def after_iteration(self, model, epoch, evals_log):
                limits.check()
                return False
        model.set_params(callbacks=[Callback()])
        model.fit(x, y, sample_weight=weights, verbose=False)
        model.set_params(callbacks=None)
    else:
        def callback(env):
            limits.check()
        callback.order = 5
        model.fit(x, y, sample_weight=weights, callbacks=[callback])
    limits.check(force=True)
    return model

```

### `training.fit_final_models` — kode aktual

**File:** `KODE_ML/research/training.py`.

**Input:** folder run, algoritma, recipe, seluruh fitur/target train, urutan label. **Output:** `label_000.joblib` dan seterusnya serta `models.json`.

Loop `enumerate(labels)` menjaga hubungan nama label, kolom `y[:, j]`, dan file model. Checkpoint yang sudah ada dicek hash-nya sebelum dilewati. Model yang belum ada dilatih, disimpan, lalu dicatat. Jadi run terputus tidak harus mengulang semua classifier.

Sel berikut menampilkan definisi lengkap beserta lokasi dan hash sumber. Isinya dipakai oleh pipeline, bukan implementasi kedua.

In [9]:
tampilkan_sumber(training.fit_final_models)

**Source aktual:** `KODE_ML/research/training.py`, baris 55–66. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
def fit_final_models(folder, algorithm, params, x, y, labels):
    saved = read(folder / "models.json") if (folder / "models.json").exists() else {}
    for j, name in enumerate(labels):
        if name in saved:
            if digest(folder / saved[name]["file"]) != saved[name]["sha256"]:
                raise ValueError("Final-model checkpoint changed.")
            continue
        model = fit(algorithm, params, x, y[:, j], Limits())
        filename = f"label_{j:03d}.joblib"
        joblib.dump(model, folder / filename)
        saved[name] = {"file": filename, "sha256": digest(folder / filename)}
        write(folder / "models.json", saved)

```

### Periksa model final yang sudah ada

Ini pemeriksaan daftar file tersimpan. Tidak ada `.fit()` atau penulisan model. Jumlah classifier setiap kandidat harus sama dengan jumlah label.

In [10]:
rows = []
for key in experiment.CANDIDATES:
    models = runtime.read(run / key / "models.json")
    assert set(models) == set(labels)
    assert all((run / key / item["file"]).is_file() for item in models.values())
    rows.append({"kandidat": key, "classifier_tersimpan": len(models)})
display(pd.DataFrame(rows))
display(pd.DataFrame.from_dict(runtime.read(run / "lgbm_D/models.json"), orient="index").head())

,kandidat,classifier_tersimpan
0,xgb_A,109
1,xgb_B,109
2,xgb_C,109
3,xgb_D,109
4,lgbm_A,109
5,lgbm_B,109
6,lgbm_C,109
7,lgbm_D,109


,file,sha256
acidic,label_000.joblib,458271e32cb9e0229612cdeafec84a3fd732ac6434948a...
aldehydic,label_001.joblib,70eb2a22468a4b36eeef7d772e8c64d931c74e128f8649...
almond,label_002.joblib,42f5c5d5594f9a5b3cb83308abe829fc3ba6d79390f5cb...
amber,label_003.joblib,e0186a80498a8ee5c25055f0bbc393d4d26550b608e976...
animal like,label_004.joblib,610ca1d26b3221e007482ebb97bc9ff5fcffbfc00e016e...


## Bekal sidang

- **Kenapa 109 model?** Satu classifier biner untuk tiap label; beberapa classifier dapat positif untuk satu molekul.
- **n_estimators itu jumlah label?** Bukan; itu jumlah boosting round/pohon dalam satu classifier.
- **Kenapa bobot dihitung dalam fold?** Agar validation tidak ikut menentukan bobot kelas.
- **Apa yang sudah diuji?** Pemuatan train, objek estimator tanpa fit, rumus bobot, dan file model. Training ulang belum diverifikasi.

Lanjut: **06_Optuna_CV_dan_OOF.ipynb**.